# FlashAttentionCUDA — Nível 0: baseline + harness de correção

Checklist deste notebook (Nível 0 do plano de desenvolvimento, semana 1):

1. Confirmar GPU disponível (T4, tier grátis do Colab).
2. Clonar o repositório e compilar a extensão CUDA ingênua.
3. Rodar o harness de correção (kernel ingênuo vs. referência PyTorch, por tolerância numérica).
4. Confirmar empiricamente a limitação `ERR_NVGPUCTRPERM` (Nsight Compute / nvprof não funcionam aqui).
5. Confirmar que eventos CUDA funcionam como alternativa de medição de tempo.


## 1. GPU disponível

In [ ]:
!nvidia-smi

## 2. Clonar o repositório e compilar

Se o repositório `IagoNat/HPC` for privado, troque a URL abaixo por uma
com um personal access token: `https://<token>@github.com/IagoNat/HPC.git`.


In [ ]:
!git clone https://github.com/IagoNat/HPC.git
%cd HPC/FlashAttentionCUDA


In [ ]:
# compila a extensão (primeira chamada faz o build e já roda um teste mínimo)
import torch
from python.naive_attention import naive_attention

Q = torch.randn(1, 1, 8, 16, device="cuda", dtype=torch.float32)
K = torch.randn(1, 1, 8, 16, device="cuda", dtype=torch.float32)
V = torch.randn(1, 1, 8, 16, device="cuda", dtype=torch.float32)
out = naive_attention(Q, K, V)
print("compilou e rodou - shape de saída:", out.shape)


## 3. Harness de corretude

In [ ]:
!python tests/test_correctness.py

## 4. Confirmar a limitação de profiling (`ERR_NVGPUCTRPERM`)

Esperado: tanto `nvprof` quanto o Nsight Compute (`ncu`) devem falhar por
falta de acesso privilegiado ao driver, em uma VM compartilhada como a do
Colab. Isso confirma o item (d) da Seção 3 da proposta e justifica usar
eventos CUDA / `nvidia-smi` como alternativa de medição (próxima célula).


In [ ]:
!nvprof python tests/test_correctness.py


In [ ]:
!ncu --target-processes all python tests/test_correctness.py


## 5. Alternativa de medição: eventos CUDA

Confirma que `torch.cuda.Event` funciona normalmente mesmo sem acesso aos
contadores de hardware — é o que será usado no benchmark do Nível 1.


In [ ]:
import torch
from python.naive_attention import naive_attention

B, H, N, D = 2, 4, 512, 64
Q = torch.randn(B, H, N, D, device="cuda", dtype=torch.float32)
K = torch.randn(B, H, N, D, device="cuda", dtype=torch.float32)
V = torch.randn(B, H, N, D, device="cuda", dtype=torch.float32)

# aquecimento
for _ in range(3):
    naive_attention(Q, K, V)
torch.cuda.synchronize()

start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)

start.record()
for _ in range(10):
    out = naive_attention(Q, K, V)
end.record()
torch.cuda.synchronize()

ms_per_call = start.elapsed_time(end) / 10
print(f"tempo médio por chamada: {ms_per_call:.3f} ms  (N={N}, D={D})")


## Status do Nível 0

Preencher após rodar este notebook:

- [ ] Extensão CUDA compila e roda na T4.
- [ ] Harness de corretude passa em todos os casos de teste.
- [ ] `ERR_NVGPUCTRPERM` confirmado (nvprof/ncu falham).
- [ ] Eventos CUDA funcionam como alternativa de medição.
